# Topic-Level Feature Analysis

Uses Fine-tuned RoBERTa predictions (`bert_predictions.csv`, full dataset 134,068 rows)  
to compute sentiment distribution per topic.

**Output:** Total reviews, % positive, % negative, sentiment score per topic  
**Sorted:** Most praised → most criticized

In [1]:
import pandas as pd

In [2]:
# Load full predictions
df = pd.read_csv("fine-tuned bert/bert_predictions.csv", low_memory=False)
print(f"Total rows: {len(df)}")
print(f"pred_sentiment distribution:\n{df['pred_sentiment'].value_counts(dropna=False)}")

Total rows: 134068
pred_sentiment distribution:
pred_sentiment
positive    111174
negative     22894
Name: count, dtype: int64


In [3]:
# Keep only rows with a valid topic label
df_topic = df[df["topic_label"].notna()].copy()
print(f"Rows with topic label: {len(df_topic)}")
print(f"Number of topics: {df_topic['topic_label'].nunique()}")

Rows with topic label: 100248
Number of topics: 14


In [4]:
# Compute per-topic sentiment distribution
summary = (
    df_topic
    .groupby("topic_label")["pred_sentiment"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"positive": "n_pos", "negative": "n_neg"})
)

# Make sure both columns exist even if a topic has 0 of one class
for col in ["n_pos", "n_neg"]:
    if col not in summary.columns:
        summary[col] = 0

summary["total"]   = summary["n_pos"] + summary["n_neg"]
summary["pos_pct"] = summary["n_pos"] / summary["total"]
summary["neg_pct"] = summary["n_neg"] / summary["total"]
summary["score"]   = summary["pos_pct"] - summary["neg_pct"]  # range: -1 to +1

# Sort from most praised to most criticized
summary = summary.sort_values("score", ascending=False)

summary.head()

pred_sentiment,n_neg,n_pos,total,pos_pct,neg_pct,score
topic_label,,,,,,
Acoustic tone,147,4413,4560,0.967763,0.032237,0.935526
Pickups,363,7492,7855,0.953787,0.046213,0.907575
Setup / action,281,4988,5269,0.946669,0.053331,0.893338
Beginner learning,1092,12785,13877,0.921309,0.078691,0.842617
Guitar size,497,5300,5797,0.914266,0.085734,0.828532


In [5]:
# Pretty-print the results
print(f"{'Topic':<30} {'Total':>7} {'Positive':>10} {'Negative':>10} {'Score':>7}")
print("-" * 67)

for topic, row in summary.iterrows():
    score_str = f"+{row['score']:.2f}" if row['score'] >= 0 else f"{row['score']:.2f}"
    print(
        f"{topic:<30} {int(row['total']):>7,}"
        f"  {row['pos_pct']:>8.1%}"
        f"  {row['neg_pct']:>8.1%}"
        f"  {score_str:>7}"
    )

Topic                            Total   Positive   Negative   Score
-------------------------------------------------------------------
Acoustic tone                    4,560     96.8%      3.2%    +0.94
Pickups                          7,855     95.4%      4.6%    +0.91
Setup / action                   5,269     94.7%      5.3%    +0.89
Beginner learning               13,877     92.1%      7.9%    +0.84
Guitar size                      5,797     91.4%      8.6%    +0.83
Playability / chords             5,576     89.4%     10.6%    +0.79
Accessories                      9,492     86.2%     13.8%    +0.72
Visual appearance                4,166     84.2%     15.8%    +0.68
Electronics / controls           2,378     82.8%     17.2%    +0.66
Fret / neck setup               13,387     73.1%     26.9%    +0.46
String quality                   6,212     60.0%     40.0%    +0.20
Customer service / returns       9,134     57.9%     42.1%    +0.16
Tuning stability                 5,964     50.1

In [6]:
# Save to CSV
output = summary[["total", "n_pos", "n_neg", "pos_pct", "neg_pct", "score"]].copy()
output.index.name = "topic_label"
output.to_csv("topic_feature_analysis.csv")
print("Saved to topic_feature_analysis.csv")

Saved to topic_feature_analysis.csv
